## Hyperparameter tuning

This notebook runs a lightweight hyperparameter search over the neural network built in 04_neural_network.ipynb, to test whether tuning architecture and regularization settings can meaningfully improve on its initial results (ROC-AUC 0.7713, F1 0.5239), and by extension, on the Logistic Regression and Random Forest baselines. A manual grid search was used rather than an automated tool like keras_tuner, given the small, deliberately scoped search space: three architectures ([32,16], [64,32], [128,64]) × two dropout rates (0.2, 0.3), with learning rate fixed at Adam's default (0.001) and all other settings (class weighting, early stopping, loss function) held constant so any performance differences can be attributed to the parameters actually being varied.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import itertools

from src.data_loader import load_raw_data
from src.preprocessing import run_preprocessing_pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, f1_score

df = load_raw_data()
splits, artifacts = run_preprocessing_pipeline(df)

class_weights = compute_class_weight(
    'balanced', classes=np.unique(splits.y_train), y=splits.y_train
)
class_weight_dict = dict(enumerate(class_weights))


def build_and_train(hidden_units, dropout_rate, learning_rate, verbose=0):
    model = keras.Sequential([layers.Input(shape=(splits.X_train.shape[1],))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
    model.fit(
        splits.X_train, splits.y_train,
        validation_data=(splits.X_val, splits.y_val),
        epochs=50, batch_size=256, class_weight=class_weight_dict,
        callbacks=[early_stop], verbose=verbose
    )

    y_proba = model.predict(splits.X_val, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)
    return model, roc_auc_score(splits.y_val, y_proba), f1_score(splits.y_val, y_pred)


# Search space — small and deliberate, not exhaustive.
architectures = [[32, 16], [64, 32], [128, 64]]
dropout_rates = [0.2, 0.3]
learning_rates = [0.001]

results = []
for hidden_units, dropout_rate, learning_rate in itertools.product(architectures, dropout_rates, learning_rates):
    _, auc, f1 = build_and_train(hidden_units, dropout_rate, learning_rate)
    results.append({
        'hidden_units': str(hidden_units),
        'dropout_rate': dropout_rate,
        'learning_rate': learning_rate,
        'roc_auc': auc,
        'f1': f1,
    })
    print(f"{hidden_units}, dropout={dropout_rate} -> ROC-AUC={auc:.4f}, F1={f1:.4f}")

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df

Loaded data via local (30000 rows, 25 columns).
[32, 16], dropout=0.2 -> ROC-AUC=0.7791, F1=0.5319
[32, 16], dropout=0.3 -> ROC-AUC=0.7748, F1=0.5401
[64, 32], dropout=0.2 -> ROC-AUC=0.7756, F1=0.5248
[64, 32], dropout=0.3 -> ROC-AUC=0.7817, F1=0.5329
[128, 64], dropout=0.2 -> ROC-AUC=0.7742, F1=0.5318
[128, 64], dropout=0.3 -> ROC-AUC=0.7782, F1=0.5423


,hidden_units,dropout_rate,learning_rate,roc_auc,f1
3,"[64, 32]",0.3,0.001,0.781659,0.532872
0,"[32, 16]",0.2,0.001,0.779141,0.531934
5,"[128, 64]",0.3,0.001,0.778229,0.542342
2,"[64, 32]",0.2,0.001,0.775639,0.524817
1,"[32, 16]",0.3,0.001,0.774790,0.540065
4,"[128, 64]",0.2,0.001,0.774249,0.531782


#### Updated Models Comparison table 

| Model | ROC-AUC | F1 |
|---|---|---|
| Logistic Regression | 0.75 | 0.52 |
| Random Forest | 0.77 | 0.45 |
| Neural Network (untuned) | 0.77 | 0.52 |
| Neural Network (tuned, best: [64,32], dropout=0.3) | 0.78 | 0.53 |

Across all six configurations, ROC-AUC ranged narrowly from 0.774 to 0.782, and F1 from 0.525 to 0.542 — a spread of less than one percentage point between the best and worst configurations. This confirms, with direct evidence rather than a single data point, that the neural network's performance on this dataset is not sensitive to architecture size or dropout rate within the ranges tested. The best-performing configuration ([64,32] hidden units, dropout=0.3) achieved ROC-AUC 0.7817 and F1 0.5329 — a small improvement over the original untuned network (0.7713 / 0.5239) and a marginal edge over both baselines (Logistic Regression: 0.75 / 0.52, Random Forest: 0.77 / 0.45), but not a dramatic one.

Taken together with the baseline comparison, the overall finding across this project is that model complexity (Random Forest, and especially the neural network) provided only marginal gains over Logistic Regression on this dataset — consistent with the broader pattern that tabular data with a moderate number of features often doesn't reward deep learning the way unstructured data does. The tuned neural network ([64,32], dropout=0.3) is selected as the final model going into the Streamlit app, primarily for its best overall ROC-AUC and its use of TensorFlow, but this is documented as a marginal choice justified by the full comparison above, not a dramatic win.